# Text-Only Fashion Style Classification (FashionBERT)

## Objective
This notebook ports the **text-only robustness** experiment from the fusion project’s `TextOnly_Robustness_Experiments` template into the **StyleFusion** layout (same spirit as `6_Image_only.ipynb`). We fine-tune a **frozen BERT-base-uncased** encoder with an MLP head (768 → 256 → 128 → 14) on **FashionStyle14**, using **10** stratified splits from `seeds_list.txt` (the file may list more seeds; we only run the first **10**).

## Pipeline (end-to-end)
1. **Resolve paths** from the repo root or `StyleFusion/`, load seeds from `seeds_list.txt`, keep **`NUM_SEEDS_TO_USE = 10`**, set hyperparameters.
2. Check device (CPU / CUDA), VRAM, and pick a safe batch size (AMP on CUDA when available).
3. Load paths from `complete_dataset.csv`, merge **captions** from the caption CSV (see Inputs), keep rows with files on disk and non-empty text.
4. For **each seed**: stratified 70% / 15% / 15% train/val/test (`random_state=seed`), build DataLoaders, initialize the model (**model-init seed fixed at 42**), train with early stopping on **validation macro-F1**, evaluate on test, save artifacts under `results/text_only/fashionbert/seed_<seed>/`.
5. Aggregate test metrics into `all_seeds_summary.csv` and print mean ± std.
6. **Learning curves** are **displayed in the notebook** (`plt.show()`); we do **not** write PNGs (unlike the older fusion notebook’s `plt.savefig`).

## Inputs
| Input | Description |
|-------|-------------|
| `FashionStyle14_v1/complete_dataset.csv` | One relative image path per line (`dataset/<style>/...`). |
| `FashionStyle14_v1/seeds_list.txt` | Robustness seeds (`Seed N` lines); **first 10** parsed values are used. |
| `FashionStyle14_v1/fashion_captions_llava_success.csv` | Captions keyed by relative paths (set `CAPTION_CSV` in §1 if your file name differs). If a `status` column exists, only `success` rows are kept. |
| Pre-trained weights | `bert-base-uncased` (Hugging Face Transformers). |

## Outputs (on disk)
Per seed: **`results/text_only/fashionbert/seed_<seed>/`**

| File | Description |
|------|-------------|
| `best_model.pt` | Best validation macro-F1 checkpoint (head weights). |
| `training_history.json` | Per-epoch train/val loss and metrics. |
| `test_metrics.json` | Overall test accuracy, macro precision / recall / F1. |
| `per_class.csv` | Per-class acc / precision / recall / F1. |

Aggregated: **`results/text_only/fashionbert/all_seeds_summary.csv`** (one row per seed).

**Figures:** learning curves are **in-notebook only** (no image files saved).

## Notes
- Code and comments are **English** only.
- Working directory may be the **repo root** or **`StyleFusion/`**; path resolution matches the image-only notebook pattern.

## 1. Configuration, imports, and paths

In [4]:
from __future__ import annotations

import json
import os
import random
import re
import warnings
from pathlib import Path
from typing import Any, Callable, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

LEARNING_RATE = 5e-5
BATCH_SIZE = 32
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MODEL_INIT_SEED = 42
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
MAX_SEQ_LENGTH = 128

BERT_MODEL_ID = "bert-base-uncased"
NUM_SEEDS_TO_USE = 10

random.seed(MODEL_INIT_SEED)
np.random.seed(MODEL_INIT_SEED)
torch.manual_seed(MODEL_INIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(MODEL_INIT_SEED)


def load_seeds(seeds_file: Path) -> List[int]:
    """Parse all seeds from `seeds_list.txt` (same pattern as image-only notebook)."""
    if not seeds_file.is_file():
        raise FileNotFoundError(f"Missing seeds file: {seeds_file}")
    content = seeds_file.read_text(encoding="utf-8")
    matches = re.findall(r"Seed\s+(\d+)", content, flags=re.IGNORECASE)
    seeds = sorted({int(s) for s in matches if 1 <= int(s) <= 500})
    if len(seeds) != 30:
        print(f"Warning: expected 30 seeds in file, found {len(seeds)}")
    return seeds


def resolve_paths() -> Tuple[Path, Path, Path]:
    """Find project root and FashionStyle14_v1 (same idea as `6_Image_only.ipynb`)."""
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "StyleFusion",
        cwd.parent / "StyleFusion",
        cwd / "FusionStyle",
        cwd.parent / "FusionStyle",
    ]
    for root in candidates:
        data_dir = root / "FashionStyle14_v1"
        dataset_dir = data_dir / "dataset"
        has_splits = (data_dir / "train.csv").is_file() or (data_dir / "train_new.csv").is_file()
        if has_splits and dataset_dir.is_dir():
            results_root = root / "results" / "text_only"
            return root, data_dir, results_root
    raise FileNotFoundError(
        "Could not locate FashionStyle14_v1 with train (or train_new) CSV and dataset/. "
        "Open the notebook from the repo root or StyleFusion/."
    )


PROJECT_ROOT, DATA_DIR, RESULTS_ROOT = resolve_paths()
IMAGE_ROOT = DATA_DIR
SEEDS_FILE = DATA_DIR / "seeds_list.txt"
COMPLETE_CSV = DATA_DIR / "complete_dataset.csv"
CAPTION_CSV = DATA_DIR / "caption" / "fashion_captions_llava_success.csv"

for label, path, must_exist in [
    ("DATA_DIR", DATA_DIR, "dir"),
    ("FashionStyle14_v1/dataset/", DATA_DIR / "dataset", "dir"),
    ("SEEDS_FILE", SEEDS_FILE, "file"),
    ("COMPLETE_CSV", COMPLETE_CSV, "file"),
    ("CAPTION_CSV", CAPTION_CSV, "file"),
]:
    ok = path.is_dir() if must_exist == "dir" else path.is_file()
    if not ok:
        raise FileNotFoundError(f"Missing {label}: {path}")

ALL_PARSED_SEEDS = load_seeds(SEEDS_FILE)
SEEDS = ALL_PARSED_SEEDS[:NUM_SEEDS_TO_USE]
if len(SEEDS) < NUM_SEEDS_TO_USE:
    print(f"Warning: only {len(SEEDS)} seeds available; expected {NUM_SEEDS_TO_USE}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("COMPLETE_CSV:", COMPLETE_CSV)
print("CAPTION_CSV:", CAPTION_CSV)
print("RESULTS_ROOT:", RESULTS_ROOT)
print(f"Using {len(SEEDS)} seeds:", SEEDS)

PROJECT_ROOT: /home/sandy/Document/FusionStyle
DATA_DIR: /home/sandy/Document/FusionStyle/FashionStyle14_v1
COMPLETE_CSV: /home/sandy/Document/FusionStyle/FashionStyle14_v1/complete_dataset.csv
CAPTION_CSV: /home/sandy/Document/FusionStyle/FashionStyle14_v1/caption/fashion_captions_llava_success.csv
RESULTS_ROOT: /home/sandy/Document/FusionStyle/results/text_only
Using 10 seeds: [13, 14, 16, 17, 45, 48, 53, 58, 72, 102]


## 2. Hardware check and runtime settings

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    total_mem_gb = props.total_memory / (1024**3)
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {total_mem_gb:.2f} GB")
    if total_mem_gb < 8:
        BATCH_SIZE = 8
    elif total_mem_gb < 12:
        BATCH_SIZE = 16
    else:
        BATCH_SIZE = 32
else:
    print("CUDA not available; using CPU (training will be slow).")
    BATCH_SIZE = 8

print(f"Selected BATCH_SIZE: {BATCH_SIZE}")
print(f"Using AMP on CUDA: {use_amp}")

CUDA device: NVIDIA GeForce RTX 4060
Total GPU memory: 7.60 GB
Selected BATCH_SIZE: 8
Using AMP on CUDA: True


## 3. Load dataset, captions, and stratified split helper

In [7]:
def normalize_rel_path(path_str: str) -> str:
    return str(path_str).strip().replace("\\", "/")


def load_complete_dataset(csv_path: Path) -> pd.DataFrame:
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    rel = [ln.strip() for ln in lines if ln.strip()]
    df = pd.DataFrame({"rel_path": rel})
    df["rel_path"] = df["rel_path"].map(normalize_rel_path)
    df["style"] = df["rel_path"].str.split("/").str[1]
    df["abs_path"] = df["rel_path"].apply(lambda r: str((IMAGE_ROOT / r.replace("/", os.sep)).resolve()))
    df = df[df["abs_path"].map(os.path.isfile)].reset_index(drop=True)
    return df


def load_captions_long(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")
    work = df.copy()
    if "status" in work.columns:
        work = work[work["status"].astype(str).str.lower() == "success"]
    path_col = None
    for c in work.columns:
        cl = c.lower().strip()
        if cl in {"image_path", "rel_path", "path", "filename"}:
            path_col = c
            break
    if path_col is None:
        raise ValueError(f"Caption CSV needs an image path column; got columns {list(work.columns)}")
    cap_col = None
    for c in work.columns:
        if c.lower().strip() in {"caption", "text", "description"}:
            cap_col = c
            break
    if cap_col is None:
        raise ValueError(f"Caption CSV needs a caption column; got columns {list(work.columns)}")
    out = work[[path_col, cap_col]].rename(columns={path_col: "rel_path", cap_col: "caption"})
    out["rel_path"] = out["rel_path"].map(normalize_rel_path)
    out["caption"] = out["caption"].fillna("").astype(str).str.strip()
    out = out[out["caption"] != ""].drop_duplicates(subset=["rel_path"], keep="last")
    return out.reset_index(drop=True)


# df_paths = load_complete_dataset(COMPLETE_CSV)
# cap_df = load_captions_long(CAPTION_CSV)
# df_full = df_paths.merge(cap_df, on="rel_path", how="inner").reset_index(drop=True)

# classes = sorted(df_full["style"].unique().tolist())
# assert len(classes) == 14, f"Expected 14 classes, got {len(classes)}: {classes}"
# style_to_idx = {s: i for i, s in enumerate(classes)}
# num_classes = len(classes)

# print("Samples with captions (files on disk):", len(df_full))
# print("Classes:", classes)
# print(f"Data splits will be created per seed ({len(SEEDS)} experiments — first {NUM_SEEDS_TO_USE} seeds from file).")

df_full = load_complete_dataset(COMPLETE_CSV)
classes = sorted(df_full["style"].unique().tolist())
assert len(classes) == 14, f"Expected 14 classes, got {len(classes)}: {classes}"
style_to_idx = {s: i for i, s in enumerate(classes)}
num_classes = len(classes)

print("Full dataset size (existing files only):", len(df_full))
print("Number of classes:", num_classes)
print("Classes:", classes)
print("Data splits will be created per seed (10 experiments per model).")


def split_by_seed(df: pd.DataFrame, seed_value: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        df,
        test_size=(VAL_RATIO + TEST_RATIO),
        stratify=df["style"],
        random_state=seed_value,
    )
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify=temp_df["style"],
        random_state=seed_value,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

Full dataset size (existing files only): 13212
Number of classes: 14
Classes: ['conservative', 'dressy', 'ethnic', 'fairy', 'feminine', 'gal', 'girlish', 'kireime-casual', 'lolita', 'mode', 'natural', 'retro', 'rock', 'street']
Data splits will be created per seed (10 experiments per model).


## 4. PyTorch dataset and DataLoaders

In [8]:
class FashionTextDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, style_to_idx: Dict[str, int]):
        self.frame = frame.reset_index(drop=True)
        self.style_to_idx = style_to_idx

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.frame.iloc[idx]
        return {"caption": str(row["caption"]), "label": self.style_to_idx[row["style"]]}


NUM_WORKERS = 0


def make_loaders(
    train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    train_ds = FashionTextDataset(train_df, style_to_idx)
    val_ds = FashionTextDataset(val_df, style_to_idx)
    test_ds = FashionTextDataset(test_df, style_to_idx)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    return train_loader, val_loader, test_loader


print("Dataset class ready. Batch size:", BATCH_SIZE)

Dataset class ready. Batch size: 8


## 5. Model: frozen BERT + MLP head

In [9]:
class TextOnlyFashionClassifier(nn.Module):
    """Frozen BERT [CLS] features (768) + head 768→256→128→num_classes (robustness reference)."""

    def __init__(self, bert: nn.Module, tokenizer: AutoTokenizer, num_classes: int, dropout: float = DROPOUT):
        super().__init__()
        self.bert = bert
        self.tokenizer = tokenizer
        for p in self.bert.parameters():
            p.requires_grad = False
        self.bert.eval()
        self.head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, captions: List[str]) -> torch.Tensor:
        device = next(self.head.parameters()).device
        with torch.no_grad():
            enc = self.tokenizer(
                captions,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to(device)
            out = self.bert(**enc)
            cls = out.last_hidden_state[:, 0, :].float()
        return self.head(cls)


# Backbone + tokenizer are constructed fresh inside build_model() each seed (clean submodule graph).


def build_model() -> TextOnlyFashionClassifier:
    bert = AutoModel.from_pretrained(BERT_MODEL_ID)
    tok = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    return TextOnlyFashionClassifier(bert, tok, num_classes=num_classes)


print("Model class ready; call build_model() each seed (fresh BERT submodule graph).")

Model class ready; call build_model() each seed (fresh BERT submodule graph).


## 6. Training, evaluation, and in-notebook learning curves

In [10]:
def evaluate(
    model: nn.Module, loader: DataLoader, criterion, device: torch.device, use_amp: bool
) -> Tuple[float, Dict[str, float], np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_preds: List[int] = []
    all_labels: List[int] = []
    n = len(loader.dataset)
    with torch.no_grad():
        for batch in loader:
            caps = batch["caption"]
            y = batch["label"].to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=use_amp and device.type == "cuda"):
                logits = model(caps)
                loss = criterion(logits, y)
            total_loss += float(loss.item()) * y.size(0)
            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(y.cpu().numpy().tolist())
    avg_loss = total_loss / max(n, 1)
    acc = accuracy_score(all_labels, all_preds)
    macro_p = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_r = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    metrics = {"loss": avg_loss, "accuracy": acc, "macro_precision": macro_p, "macro_recall": macro_r, "macro_f1": macro_f1}
    return avg_loss, metrics, np.array(all_preds), np.array(all_labels)


def train_one_epoch(model, loader, optimizer, criterion, device, use_amp) -> Tuple[float, float]:
    model.train()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp and device.type == "cuda")
    total_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        caps = batch["caption"]
        y = batch["label"].to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp and device.type == "cuda"):
            logits = model(caps)
            loss = criterion(logits, y)
        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * y.size(0)
        preds = logits.argmax(dim=1)
        correct += int((preds == y).sum().item())
        total += int(y.size(0))
    return total_loss / max(total, 1), correct / max(total, 1)


def plot_learning_curves(history: Dict[str, Any], title: str) -> None:
    """Display learning curves in the notebook only (no PNG written; contrasts old fusion notebook savefig)."""
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()

    axes[1].plot(epochs, history["val_macro_f1"], label="val macro F1")
    axes[1].plot(epochs, history["train_acc"], label="train acc", alpha=0.7)
    axes[1].plot(epochs, history["val_acc"], label="val acc", alpha=0.7)
    axes[1].set_title("Metrics")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def set_model_init_seed() -> None:
    random.seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    torch.manual_seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)


def train_text_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    train_df: pd.DataFrame,
    out_dir: Path,
    model_name: str,
    seed_value: int,
    device: torch.device,
    use_amp: bool,
    plot_curves: bool = True,
) -> Dict[str, Any]:
    out_dir.mkdir(parents=True, exist_ok=True)

    y_train = train_df["style"].map(style_to_idx).values
    class_weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=y_train)
    cw = torch.FloatTensor(class_weights).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )

    history = {"train_loss": [], "val_loss": [], "val_macro_f1": [], "train_acc": [], "val_acc": []}
    best_f1 = -1.0
    best_state = None
    patience_left = EARLY_STOPPING_PATIENCE

    for epoch in range(1, MAX_EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device, use_amp)
        val_loss, val_metrics, _, _ = evaluate(model, val_loader, criterion, device, use_amp)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["val_macro_f1"].append(val_metrics["macro_f1"])
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_metrics["accuracy"])

        print(
            f"Epoch {epoch:02d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_metrics['accuracy']:.4f} macroF1 {val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOPPING_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                print("Early stopping triggered.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    torch.save(model.state_dict(), out_dir / "best_model.pt")
    with open(out_dir / "training_history.json", "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)

    _, test_metrics, test_preds, test_labels = evaluate(model, test_loader, criterion, device, use_amp)

    per_class_precision = precision_score(test_labels, test_preds, average=None, zero_division=0)
    per_class_recall = recall_score(test_labels, test_preds, average=None, zero_division=0)
    per_class_f1 = f1_score(test_labels, test_preds, average=None, zero_division=0)

    per_class_rows = []
    for i, cls in enumerate(classes):
        per_class_rows.append(
            {
                "class": cls,
                "acc": float(per_class_recall[i]),
                "precision": float(per_class_precision[i]),
                "recall": float(per_class_recall[i]),
                "f1": float(per_class_f1[i]),
            }
        )
    per_class_df = pd.DataFrame(per_class_rows, columns=["class", "acc", "precision", "recall", "f1"])

    with open(out_dir / "test_metrics.json", "w", encoding="utf-8") as f:
        json.dump({k: float(v) for k, v in test_metrics.items()}, f, indent=2)
    per_class_csv_path = out_dir / "per_class.csv"
    per_class_df.to_csv(per_class_csv_path, index=False, encoding="utf-8")

    print(f"\n[{model_name} | seed {seed_value}] Test metrics:", test_metrics)
    print(f"[{model_name} | seed {seed_value}] Per-class metrics ({per_class_csv_path}):")
    print(per_class_df.to_string(index=False))

    if plot_curves:
        plot_learning_curves(history, title=f"Learning curves — {model_name} (seed {seed_value})")

    return {
        "seed": seed_value,
        "history": history,
        "test_metrics": test_metrics,
        "per_class_csv": str(per_class_csv_path),
        "out_dir": str(out_dir),
    }


def run_robustness_text(model_name: str, build_model_fn: Callable[[], nn.Module]) -> pd.DataFrame:
    model_root = RESULTS_ROOT / model_name
    model_root.mkdir(parents=True, exist_ok=True)
    rows: List[Dict[str, Any]] = []

    for seed_idx, seed_value in enumerate(SEEDS, start=1):
        seed_dir = model_root / f"seed_{seed_value}"
        done_marker = seed_dir / "test_metrics.json"
        if done_marker.is_file():
            print(f"[{model_name}] Seed {seed_value} ({seed_idx}/{len(SEEDS)}): skip (already done)")
            with open(done_marker, encoding="utf-8") as f:
                cached = json.load(f)
            rows.append({"seed": seed_value, **cached})
            continue

        print("=" * 70)
        print(f"[{model_name}] Experiment {seed_idx}/{len(SEEDS)} | data split seed = {seed_value}")
        print("=" * 70)

        train_df, val_df, test_df = split_by_seed(df_full, seed_value)
        print(f"  Split sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

        train_loader, val_loader, test_loader = make_loaders(train_df, val_df, test_df)

        set_model_init_seed()
        model = build_model_fn().to(device)

        result = train_text_model(
            model,
            train_loader,
            val_loader,
            test_loader,
            train_df,
            out_dir=seed_dir,
            model_name=model_name,
            seed_value=seed_value,
            device=device,
            use_amp=use_amp,
            plot_curves=True,
        )
        rows.append({"seed": seed_value, **{k: float(v) for k, v in result["test_metrics"].items()}})

        del model, train_loader, val_loader, test_loader
        if device.type == "cuda":
            torch.cuda.empty_cache()

    summary_df = pd.DataFrame(rows)
    summary_path = model_root / "all_seeds_summary.csv"
    summary_df.to_csv(summary_path, index=False, encoding="utf-8")
    print(f"\n[{model_name}] Saved summary: {summary_path}")
    if len(summary_df) > 0:
        print(
            f"[{model_name}] macro_f1 mean +/- std: "
            f"{summary_df['macro_f1'].mean():.4f} +/- {summary_df['macro_f1'].std():.4f}"
        )
    return summary_df


print("Training utilities ready.")

Training utilities ready.


## 7. FashionBERT: text-only robustness (10 seeds)

In [12]:
summary_bert = run_robustness_text(model_name="fashionbert", build_model_fn=build_model)

[fashionbert] Experiment 1/10 | data split seed = 13
  Split sizes: train=9248, val=1982, test=1982


/home/sandy/miniconda3/envs/fusionstyle/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


KeyError: 'caption'

## 8. Aggregate comparison (10 seeds)

In [ ]:
metric_cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]


def summarize_seeds(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    rows = []
    for col in metric_cols:
        if col not in df.columns:
            continue
        rows.append(
            {
                "model": model_name,
                "metric": col,
                "mean": float(df[col].mean()),
                "std": float(df[col].std()),
                "min": float(df[col].min()),
                "max": float(df[col].max()),
            }
        )
    return pd.DataFrame(rows)


summary_path = RESULTS_ROOT / "fashionbert" / "all_seeds_summary.csv"
print("Per-seed summary:", summary_path)

if summary_path.is_file():
    bert_df = pd.read_csv(summary_path)
    agg = summarize_seeds(bert_df, "fashionbert")
    print("\nTest metrics across seeds (mean +/- std):")
    print(agg.to_string(index=False))
else:
    print("Run section 7 first to generate all_seeds_summary.csv.")